[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nocapchicken/nocapchicken.github.io/blob/fix/audit-findings/notebooks/train_distilbert_colab.ipynb)

# DistilBERT Training — Google Colab

Run this notebook on Colab with a **T4 GPU** (Runtime → Change runtime type → T4 GPU). Expected time: ~10–15 minutes.

**Binary classification:** A (safe) vs Flagged (B or C inspection grade). Only rows with review text are used for training (~111K of 231K total).

**Setup:** Add a Colab secret named `GITHUB_TOKEN_NOCAPCHICKEN` (key icon in left sidebar) with repo write access. The notebook will clone the repo, train, and push model artifacts back automatically.

In [ ]:
# Verify GPU is available
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected — check Runtime type')

In [ ]:
%pip install -q transformers datasets scikit-learn pandas torch

In [ ]:
import subprocess, sys, os, importlib.util, shutil
from pathlib import Path

REPO = Path('/content/nocapchicken.github.io')
BRANCH = 'fix/audit-findings'

# Clean slate — avoids stale state from previous runs
if REPO.exists():
    shutil.rmtree(REPO)

env = {**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}
subprocess.run(['git', 'clone', '-b', BRANCH,
    'https://github.com/nocapchicken/nocapchicken.github.io.git',
    str(REPO)], check=True, env=env)

os.chdir(REPO)

spec = importlib.util.spec_from_file_location('colab_utils', REPO / 'scripts' / 'colab_utils.py')
colab_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_utils)
publish_artifacts = colab_utils.publish_artifacts

print(f'Repo ready at {REPO} (branch: {BRANCH})')

In [ ]:
!python scripts/build_features.py

import pandas as pd

df = pd.read_csv('data/processed/features.csv')
print(f'Total rows: {len(df)}')
print('Grade distribution:')
print(df['grade'].value_counts())

# Only train on rows with actual review text
has_text = df['combined_reviews'].fillna('').str.strip().ne('')
df = df[has_text].copy()
print(f'\nRows with review text (used for training): {len(df)}')

# Binary target: A=0 (safe), B+C=1 (flagged)
df['binary_label'] = (df['grade_encoded'] > 0).astype(int)
print(f'Label distribution: {df["binary_label"].value_counts().to_dict()}')

In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TEST_SIZE = 0.2

texts = df['combined_reviews'].tolist()
labels = df['binary_label'].tolist()

texts_train, texts_test, y_train, y_test = train_test_split(
    texts, labels,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=labels,
)

print(f'Train: {len(texts_train)}  Test: {len(texts_test)}  Binary: safe vs flagged')

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)
import numpy as np
from sklearn.metrics import classification_report


class ReviewDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)


tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
train_enc = tokenizer(texts_train, truncation=True, padding=True, max_length=256)
test_enc  = tokenizer(texts_test,  truncation=True, padding=True, max_length=256)

train_dataset = ReviewDataset(train_enc, y_train)
test_dataset  = ReviewDataset(test_enc,  y_test)

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=2  # 0=safe, 1=flagged
)

training_args = TrainingArguments(
    output_dir='./distilbert',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    logging_dir='./distilbert_logs',
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

In [ ]:
# Evaluate on test set
preds_output = trainer.predict(test_dataset)
y_pred = np.argmax(preds_output.predictions, axis=1)

label_names = ['A (safe)', 'Flagged (B/C)']
print(classification_report(y_test, y_pred, target_names=label_names))

In [ ]:
# Save model + tokenizer, then push to GitHub
trainer.save_model('models/distilbert')
tokenizer.save_pretrained('models/distilbert')

publish_artifacts(
    ['models/distilbert'],
    'feat: retrain DistilBERT binary classifier on 231K rows [Colab]',
    branch='fix/audit-findings',
)